# Emotion Mapping for Song Lyrics

This notebook maps emotions to cleaned song lyrics using RoBERTa-large fine-tuned on GoEmotions dataset.

## Process:
1. Load cleaned lyrics dataset from Google Drive
2. Use pretrained RoBERTa model to predict 27 emotions
3. Apply weighted average across lines (weight = 1 - p(neutral))
4. Aggregate emotions for each song
5. Save results to CSV

## 1. Install Required Packages

In [ ]:
!pip install transformers torch pandas numpy tqdm

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Load Pretrained RoBERTa-GoEmotions Model

In [ ]:
# Load RoBERTa model fine-tuned on GoEmotions (27 emotions)
model_name = "SamLowe/roberta-base-go_emotions"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model = model.to(device)
model.eval()

# Get emotion labels
emotion_labels = list(model.config.id2label.values())
print(f"Number of emotions: {len(emotion_labels)}")
print(f"Emotions: {emotion_labels}")

# Find neutral emotion index
neutral_idx = emotion_labels.index('neutral') if 'neutral' in emotion_labels else None
print(f"Neutral emotion index: {neutral_idx}")

## 5. Define Emotion Mapping Functions

In [ ]:
def predict_emotions(text, max_length=512):
    """
    Predict emotions for a given text using the model.
    Returns a numpy array of probabilities for all 27 emotions.
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, 
                      max_length=max_length, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    return probs.cpu().numpy()[0]


def weighted_average_emotions(emotion_vectors, neutral_idx):
    """
    Calculate weighted average of emotion vectors.
    Formula: p_song = Σ(w_i * p_i) / Σ(w_i)
    where w_i = 1 - p(neutral)
    
    Args:
        emotion_vectors: List of emotion probability vectors (each 27x1)
        neutral_idx: Index of neutral emotion in the vector
    
    Returns:
        Aggregated emotion vector (27x1)
    """
    if len(emotion_vectors) == 0:
        return np.zeros(27)
    
    emotion_vectors = np.array(emotion_vectors)
    
    # Calculate weights: w_i = 1 - p(neutral)
    if neutral_idx is not None:
        weights = 1 - emotion_vectors[:, neutral_idx]
    else:
        weights = np.ones(len(emotion_vectors))
    
    # Avoid division by zero
    weights = weights.reshape(-1, 1)
    total_weight = np.sum(weights)
    
    if total_weight == 0:
        # If all weights are zero, return uniform distribution
        return np.ones(27) / 27
    
    # Weighted average: Σ(w_i * p_i) / Σ(w_i)
    weighted_sum = np.sum(weights * emotion_vectors, axis=0)
    weighted_avg = weighted_sum / total_weight
    
    return weighted_avg


def map_lyrics_to_emotion(lyrics, neutral_idx):
    """
    Map a song's lyrics to emotions by:
    1. Splitting lyrics into lines
    2. Getting emotion vector for each line
    3. Applying weighted average across all lines
    
    Args:
        lyrics: String containing the song lyrics
        neutral_idx: Index of neutral emotion
    
    Returns:
        Aggregated emotion vector (27x1) for the entire song
    """
    if pd.isna(lyrics) or lyrics.strip() == "":
        return np.zeros(27)
    
    # Split lyrics into lines
    lines = [line.strip() for line in lyrics.split('\n') if line.strip()]
    
    if len(lines) == 0:
        return np.zeros(27)
    
    # Get emotion vector for each line
    line_emotions = []
    for line in lines:
        if line:  # Skip empty lines
            emotion_vector = predict_emotions(line)
            line_emotions.append(emotion_vector)
    
    # Aggregate emotions using weighted average
    if len(line_emotions) > 0:
        aggregated_emotion = weighted_average_emotions(line_emotions, neutral_idx)
    else:
        aggregated_emotion = np.zeros(27)
    
    return aggregated_emotion

## 6. Load Dataset from Google Drive

In [ ]:
# Update this path to your dataset location in Google Drive
data_path = '/content/drive/MyDrive/song_lyrics_cleaned.csv'

# Load the dataset
print("Loading dataset...")
df = pd.read_csv(data_path)
print(f"Dataset loaded! Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 7. Process Dataset Chunk by Chunk

In [ ]:
# Configuration
CHUNK_SIZE = 100  # Process 100 songs at a time
CLEANED_LYRICS_COLUMN = 'cleaned_lyrics'  # Update if your column name is different

# Verify the column exists
if CLEANED_LYRICS_COLUMN not in df.columns:
    print(f"Error: Column '{CLEANED_LYRICS_COLUMN}' not found!")
    print(f"Available columns: {df.columns.tolist()}")
    raise ValueError(f"Please update CLEANED_LYRICS_COLUMN to match your dataset")

# Initialize results storage
all_emotion_vectors = []

# Process in chunks
num_chunks = (len(df) + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Processing {len(df)} songs in {num_chunks} chunks...")

for chunk_idx in range(num_chunks):
    start_idx = chunk_idx * CHUNK_SIZE
    end_idx = min((chunk_idx + 1) * CHUNK_SIZE, len(df))
    
    print(f"\nProcessing chunk {chunk_idx + 1}/{num_chunks} (songs {start_idx} to {end_idx-1})...")
    
    chunk_emotions = []
    
    # Process each song in the chunk
    for idx in tqdm(range(start_idx, end_idx), desc=f"Chunk {chunk_idx + 1}"):
        lyrics = df.iloc[idx][CLEANED_LYRICS_COLUMN]
        emotion_vector = map_lyrics_to_emotion(lyrics, neutral_idx)
        chunk_emotions.append(emotion_vector)
    
    all_emotion_vectors.extend(chunk_emotions)
    print(f"Chunk {chunk_idx + 1} completed. Total processed: {len(all_emotion_vectors)}")

print(f"\nAll chunks processed! Total songs: {len(all_emotion_vectors)}")

## 8. Create Final DataFrame with Emotion Columns

In [ ]:
# Convert emotion vectors to dataframe
emotion_array = np.array(all_emotion_vectors)
emotion_df = pd.DataFrame(emotion_array, columns=emotion_labels)

print(f"Emotion DataFrame shape: {emotion_df.shape}")
print(f"\nEmotion columns: {emotion_df.columns.tolist()}")
print(f"\nFirst few rows of emotions:")
print(emotion_df.head())

# Create final dataframe with cleaned_lyrics and emotion columns
result_df = pd.DataFrame()
result_df['cleaned_lyrics'] = df[CLEANED_LYRICS_COLUMN].values
result_df = pd.concat([result_df, emotion_df], axis=1)

print(f"\nFinal DataFrame shape: {result_df.shape}")
print(f"Columns: {result_df.columns.tolist()}")
result_df.head()

## 9. Save Results to CSV

In [ ]:
# Define output path (update to your preferred location)
output_path = '/content/drive/MyDrive/lyrics_with_emotions.csv'

# Save to CSV
result_df.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")
print(f"File size: {result_df.shape}")
print(f"Number of columns: {len(result_df.columns)}")
print(f"  - 1 column for cleaned_lyrics")
print(f"  - {len(emotion_labels)} columns for emotions")

## 10. Verify Results and Statistics

In [ ]:
# Verify the data
print("=== VERIFICATION ===")
print(f"\nTotal songs processed: {len(result_df)}")
print(f"\nSample of the final dataset:")
print(result_df.head(3))

# Calculate and display emotion statistics
print("\n=== EMOTION STATISTICS ===")
print("\nMean emotion probabilities across all songs:")
emotion_means = result_df[emotion_labels].mean().sort_values(ascending=False)
print(emotion_means)

# Find top emotions for each song
print("\n=== TOP 3 EMOTIONS FOR FIRST 5 SONGS ===")
for i in range(min(5, len(result_df))):
    song_emotions = result_df.iloc[i][emotion_labels]
    top_3 = song_emotions.nlargest(3)
    print(f"\nSong {i+1}:")
    for emotion, prob in top_3.items():
        print(f"  {emotion}: {prob:.4f}")
    
print("\n✅ Emotion mapping completed successfully!")